# 11 Genre Prediction from Spotify Audio Features

This notebook builds genre classifier und predicter. This will be used to evaluate the recommender models built in other notebooks.

In [1]:
from pathlib import Path
import sys
import warnings

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

processed_dir = PROJECT_ROOT / 'data' / 'processed'

tracks = pd.read_parquet(processed_dir / 'tracks_clean.parquet')
artists = pd.read_parquet(processed_dir / 'artists_clean.parquet')
audio_features = pd.read_parquet(processed_dir / 'audio_features_clean.parquet')

print(len(tracks), len(artists), len(audio_features))

95977 56129 95948


## Merge datasets

In [2]:
"""
Merge tracks + artists, then train a classifier to fill in missing genres.

The core problem with naive merges on this kind of data:
- tracks.artists_id is a STRING that looks like a list, e.g. "['3mxJ...']"
- artists.genres is also a stringified list, e.g. "['pop', 'rock']"
- A track can have multiple artists, each with different genres
Pandas treats both as plain text unless you explicitly parse them, so a
direct merge on artists_id == artists.id will never match anything.

Pipeline:
1. Parse stringified lists into real Python lists (ast.literal_eval).
2. Explode tracks so there's one row per (track, artist) pair.
3. Merge that against artists on the artist id.
4. Re-aggregate genres back up to one row per track (union of all its
   artists' genres), since a track can have N artists.
5. Split into "has genre" (training data) vs "empty genre" (rows to predict).
6. Train a multi-label classifier (a track can have multiple genres) using
   audio/track features, then predict genres for the empty rows.
"""

import ast
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report

# ---------- Step 1: parse stringified lists ----------
def parse_list_string(s):
    """Turn "['pop', 'rock']" into ['pop', 'rock']. Handles NaN and junk safely."""
    if pd.isna(s):
        return []
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return [v for v in val if v]  # drop empty strings inside the list
        return []
    except (ValueError, SyntaxError):
        return []


tracks["artist_id_list"] = tracks["artists_id"].apply(parse_list_string)
artists["genre_list"] = artists["genres"].apply(parse_list_string)

# Sanity check before continuing -- if this prints mostly 0s/1s something
# upstream is still wrong (e.g. wrong column, or not actually list-strings).
print("Tracks with at least one parsed artist id:",
      (tracks["artist_id_list"].apply(len) > 0).mean())
print("Artists with at least one parsed genre:",
      (artists["genre_list"].apply(len) > 0).mean())


# ---------- Step 2: explode tracks to one row per (track, artist) ----------
exploded = tracks.explode("artist_id_list").rename(columns={"artist_id_list": "artist_id"})


# ---------- Step 3: merge against artists on artist id ----------
# Use artists' real id column, not its track_id column -- track_id is for a
# different relationship (artist -> their tracks) and isn't the join key here.
merged = exploded.merge(
    artists[["id", "genre_list"]],
    left_on="artist_id",
    right_on="id",
    how="left",
    suffixes=("", "_artist"),
)


# ---------- Step 4: re-aggregate to one row per track ----------
agg_genres = (
    merged.groupby("id")["genre_list"]
    .apply(lambda lists: sorted(set(g for sub in lists for g in (sub if isinstance(sub, list) else []))))
)

tracks_final = tracks.drop(columns=["artist_id_list"]).merge(
    agg_genres.rename("genre_list"), on="id", how="left"
)

print(f"\nTotal tracks: {len(tracks_final)}")
print(f"Tracks with at least one genre: {(tracks_final['genre_list'].apply(len) > 0).sum()}")
print(f"Tracks with empty genre: {(tracks_final['genre_list'].apply(len) == 0).sum()}")

Tracks with at least one parsed artist id: 1.0
Artists with at least one parsed genre: 0.5806089543729623

Total tracks: 95977
Tracks with at least one genre: 80448
Tracks with empty genre: 15529


## Mapping
To get the number of different genres down and to make training a prediction model easier, we map subgenres into bigger groups.

In [16]:
import re

def normalize(g):
    g = str(g).lower()
    g = g.replace("-", " ")
    g = re.sub(r"[^a-z0-9\s]", "", g)
    g = re.sub(r"\s+", " ", g).strip()
    return g

In [12]:
ROOTS = {
    "hip_hop": ["hip hop", "rap", "trap", "drill", "grime", "hiphop"],
    "rock": ["rock", "alt rock", "indie rock", "alternative rock", "punk"],
    "pop": ["pop", "synth pop", "indie pop", "electropop", "alt pop"],
    "electronic": ["electronic", "edm", "house", "techno", "trance", "dubstep", "dnb"],
    "metal": ["metal", "death metal", "black metal", "metalcore", "djent"],
    "jazz": ["jazz", "fusion", "bebop"],
    "classical": ["classical piano", "orchestra", "symphony"],
    "rnb": ["rnb", "r&b", "soul", "neo soul", "funk"],
    "latin": ["latin", "reggaeton", "bachata", "salsa"],
    "country": ["country", "americana", "bluegrass"],
    "folk": ["folk", "acoustic", "singer songwriter"],
    "lo-fi": ["lo-fi beats", "lofi", "lo-fi", "lo fi beats"]
}

In [17]:
def map_to_root(genres):
    if not isinstance(genres, list):
        return ["other"]

    genres = [normalize(g) for g in genres]

    result = set()

    for g in genres:
        matched = False

        for root, keywords in ROOTS.items():
            for k in keywords:
                if k in g:
                    result.add(root)
                    matched = True
                    break
            if matched:
                break

        if not matched:
            # DON'T immediately go to "other"
            # try weak fallback assignment first
            result.add("unknown_candidate")

    # final cleanup
    if len(result) == 0:
        return ["other"]

    # if everything failed, only then "other"
    if result == {"unknown_candidate"}:
        return ["other"]

    return list(result - {"unknown_candidate"})

In [18]:
def fallback_guess(g):
    if "rock" in g: return "rock"
    if "rap" in g or "hip" in g: return "hip_hop"
    if "pop" in g: return "pop"
    if "house" in g or "techno" in g: return "electronic"
    return "other"

In [19]:
# ---------- Step 5: normalize + map genres first ----------
tracks_final["genre_list"] = tracks_final["genre_list"].apply(
    lambda x: x if isinstance(x, list) else []
)
tracks_final["genre_canonical"] = tracks_final["genre_list"].apply(map_to_root)

has_genre = tracks_final[
    tracks_final["genre_canonical"].apply(lambda x: len(x) > 0 and "other" not in x)
].copy()
missing_genre = tracks_final[
    tracks_final["genre_canonical"].apply(lambda x: len(x) == 0 or x == ["other"])
].copy()

In [ ]:
# ---------- Step 5: split into labeled / unlabeled ----------
has_genre = tracks_final[tracks_final["genre_list"].apply(len) > 0].copy()
missing_genre = tracks_final[tracks_final["genre_list"].apply(len) == 0].copy()

has_genre["genre_canonical"] = has_genre["genre_list"].apply(map_to_canonical)

In [20]:
# ---------- Step 6: train a multi-label classifier ----------
# Pick whatever numeric features you actually have -- these are typical
# audio-feature-style columns if you bring in audio_features later via a
# similar merge. Replace with your real column names.
feature_cols = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo",
]
feature_cols = [c for c in feature_cols if c in has_genre.columns]

if not feature_cols:
    raise ValueError(
        "No usable numeric feature columns found in tracks_final. "
        "Merge in audio_features first (same explode/merge pattern, but "
        "audio_features is usually already one-row-per-track so it's a "
        "plain merge on track id, no exploding needed), then update "
        "feature_cols above."
    )

X = has_genre[feature_cols].fillna(0)
y_raw = has_genre["genre_canonical"]

mlb = MultiLabelBinarizer()

In [21]:
y = mlb.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [22]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

clf = OneVsRestClassifier(
    LogisticRegression(
        solver="saga",
        max_iter=1000,
        n_jobs=-1
    )
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("\nClassification report on held-out labeled data:")
print(classification_report(y_test, y_pred, target_names=mlb.classes_, zero_division=0))


Classification report on held-out labeled data:
              precision    recall  f1-score   support

   classical       0.73      0.61      0.67       305
     country       0.00      0.00      0.00       662
  electronic       0.72      0.21      0.32      1648
        folk       0.00      0.00      0.00      1003
     hip_hop       0.73      0.39      0.51      2739
        jazz       0.23      0.00      0.01       680
       latin       0.00      0.00      0.00      1215
       lo-fi       1.00      0.02      0.03       114
       metal       0.68      0.15      0.24       542
         pop       0.63      0.80      0.71      6811
         rnb       0.00      0.00      0.00       959
        rock       0.64      0.25      0.36      3853

   micro avg       0.65      0.40      0.49     20531
   macro avg       0.45      0.20      0.24     20531
weighted avg       0.53      0.40      0.41     20531
 samples avg       0.56      0.43      0.46     20531



In [29]:
# ---------- Step 7: predict genres for the rows that had none ----------
X_missing = missing_genre[feature_cols].fillna(0)
if len(X_missing) > 0:
    pred = clf.predict(X_missing)
    missing_genre["predicted_genre_list"] = list(mlb.inverse_transform(pred))
else:
    missing_genre["predicted_genre_list"] = []

print("\nSample of predicted genres for previously-empty rows:")
print(missing_genre[["id", "predicted_genre_list"]].head(10))

# ---------- Step 8: stitch it all back together ----------
has_genre["predicted_genre_list"] = has_genre["genre_list"]  # already known, no need to predict
tracks_with_predicted_genres = pd.concat([has_genre, missing_genre], ignore_index=True)
tracks_with_predicted_genres.to_parquet(
    processed_dir / "tracks_with_predicted_genres.parquet",
    compression="snappy",
    index=False
)


Sample of predicted genres for previously-empty rows:
                        id predicted_genre_list
0   5qljLQuKnNJf4F4vfxQB0V               (pop,)
11  2mLYN7Hz2czFkZugFscrrb               (pop,)
15  2cCTwi9uLomFcpWF0IaJW5               (pop,)
16  3cvsKLJDhmB19Oi1cPjPlB               (pop,)
18  6SO4EIIj2GPGPaX5KeWhoH               (pop,)
22  6x8r1YPztsFjyty1wRKVUG           (hip_hop,)
26  2iE7T5nZVaKoFcaXVz6M01               (pop,)
27  4p1UKkQQOeRafwDg53JAxa           (hip_hop,)
28  7hmDplqSftS7uQMxsEY9BU               (pop,)
29  3FaeRwOahQ8DyY1fSg6HO1       (hip_hop, pop)
